# Genre Classification CNN - Google Colab Training Notebook

This notebook provides a complete environment setup and execution pipeline to train the `GenreClassifierCNN` model on Google Colab using GPU acceleration.

### Workflow Overview:
1. **Environment Setup**: Check GPU status and install packages.
2. **Codebase Access**: Load your project code (via Google Drive mount, ZIP upload, or GitHub clone).
3. **Choose Training Pipeline**: Run either **Section 3 (MTG-Jamendo)** or **Section 4 (GTZAN)** to download and train.
4. **Cross-Dataset Generalization Test**: Evaluate each model on the other dataset to measure cross-dataset performance.

## 1. Environment Setup
First, verify that a GPU is available for acceleration and install required dependencies.

In [ ]:
# Verify GPU accessibility
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

In [ ]:
# Install dependencies (Colab pre-installs torch/librosa, but make sure tqdm/pandas are ready)
!pip install tqdm pandas numpy librosa torch kagglehub

## 2. Load the Codebase

Choose **ONE** of the options below to load the codebase into Colab:

### Option A: Mount Google Drive
Mount your Google Drive and navigate to your workspace folder using the defined `DRIVE_PATH` variable.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Define your Google Drive project root path here
DRIVE_PATH = "/content/drive/MyDrive/_files/Amikom University/Semester 6/mgc_research"

# Navigate to the project directory
%cd "{DRIVE_PATH}"

### Option B: Clone from GitHub
If your repository is on GitHub, clone it directly into the Colab ephemeral storage.

In [ ]:
# Replace with your actual repository URL
# !git clone https://github.com/jpangestu/mgc_research.git
# %cd mgc_research
# DRIVE_PATH = "/content/mgc_research" # Update DRIVE_PATH to point locally

## Helper Function for Path Resolution
Run this cell once. It defines the helper function used to map your local CSV indices to the dynamically downloaded Kaggle directories.

In [ ]:
import os
import glob
import pandas as pd
import kagglehub

def resolve_paths(local_csv_path, kaggle_download_path):
    if not os.path.exists(local_csv_path):
        raise FileNotFoundError(f"Local CSV index not found at {local_csv_path}")
    
    # Read CSV to locate sample files from the local index
    df = pd.read_csv(local_csv_path)
    sample_rel = df["rel_path"].iloc[0].replace("\\", "/")
    sample_filename = os.path.basename(sample_rel)
    
    # Find the sample file in the downloaded Kaggle folder
    matching_files = glob.glob(os.path.join(kaggle_download_path, "**", sample_filename), recursive=True)
    if not matching_files:
        raise FileNotFoundError(f"Could not locate any dataset files matching '{sample_filename}' in {kaggle_download_path}")
    
    physical_path = matching_files[0].replace("\\", "/")
    if physical_path.endswith(sample_rel):
        base_dir = physical_path[:-len(sample_rel)]
    else:
        base_dir = os.path.dirname(matching_files[0])
        
    return local_csv_path, base_dir

---

## 3. Pipeline A: Train on MTG-Jamendo Subset
Run this section if you want to download and train on the **MTG-Jamendo** subset.

In [ ]:
# 1. Download dataset (public anonymous download)
JAMENDO_PATH = kagglehub.dataset_download("pangestu69/mtg-jamendo-gtzan-subset")
print("[+] MTG-Jamendo subset downloaded to:", JAMENDO_PATH)

# 2. Resolve paths dynamically using the local CSV
JAMENDO_CSV, JAMENDO_BASE = resolve_paths("data/processed/jamendo_gtzan_single_label.csv", JAMENDO_PATH)
print(f"[+] Jamendo Paths Resolved:\n  CSV: {JAMENDO_CSV}\n  Base: {JAMENDO_BASE}")

In [ ]:
# Save model weights to Google Drive to preserve checkpoints
SAVE_DIR = os.path.join(DRIVE_PATH, "models")

# 3. Run training
!python main.py \
    --csv_file "{JAMENDO_CSV}" \
    --base_dir "{JAMENDO_BASE}" \
    --save_dir "{SAVE_DIR}" \
    --epochs 15 \
    --batch_size 64 \
    --lr 0.001 \
    --num_workers 2

In [ ]:
# For resuming training process on latest best model

# !python main.py \
#     --csv_file "{JAMENDO_CSV}" \
#     --base_dir "{JAMENDO_BASE}" \
#     --save_dir "{SAVE_DIR}" \
#     --epochs 30 \
#     --batch_size 64 \
#     --lr 0.0001 \
#     --num_workers 2 \
#     --resume_path "{SAVE_DIR}/best_genre_model_jamendo.pth"

---

## 4. Pipeline B: Train on GTZAN Dataset
Run this section if you want to download and train on the **GTZAN** dataset.

In [ ]:
# 1. Download dataset (public anonymous download)
GTZAN_PATH = kagglehub.dataset_download("pangestu69/gtzan-melspectogram")
print("[+] GTZAN Mel-spectrograms downloaded to:", GTZAN_PATH)

# 2. Resolve paths dynamically using the local CSV
GTZAN_CSV, GTZAN_BASE = resolve_paths("data/processed/gtzan_single_label.csv", GTZAN_PATH)
print(f"[+] GTZAN Paths Resolved:\n  CSV: {GTZAN_CSV}\n  Base: {GTZAN_BASE}")

In [ ]:
# Save model weights to Google Drive to preserve checkpoints
SAVE_DIR = os.path.join(DRIVE_PATH, "models")

# 3. Run training
!python main.py \
    --csv_file "{GTZAN_CSV}" \
    --base_dir "{GTZAN_BASE}" \
    --save_dir "{SAVE_DIR}" \
    --epochs 15 \
    --batch_size 64 \
    --lr 0.001 \
    --num_workers 2

In [ ]:
# For resuming training process on latest best model

# !python main.py \
#     --csv_file "{GTZAN_CSV}" \
#     --base_dir "{GTZAN_BASE}" \
#     --save_dir "{SAVE_DIR}" \
#     --epochs 30 \
#     --batch_size 64 \
#     --lr 0.0001 \
#     --num_workers 2 \
#     --resume_path "{SAVE_DIR}/best_genre_model_gtzan.pth"

---

## 5. Cross-Dataset Generalization Evaluation

Evaluate how well each trained model performs when tested on the other dataset. Note: Make sure you have run both Section 3 and Section 4 downloads to populate the paths before running these evaluations.

### Option A: Test Jamendo-trained model on GTZAN dataset

In [ ]:
MODEL_PATH = os.path.join(DRIVE_PATH, "models", "best_genre_model_jamendo.pth")

# Run cross evaluation
!python scripts/cross_evaluate.py \
    --model_path "{MODEL_PATH}" \
    --csv_file "{GTZAN_CSV}" \
    --base_dir "{GTZAN_BASE}" \
    --batch_size 64 \
    --num_workers 2

### Option B: Test GTZAN-trained model on Jamendo dataset

In [ ]:
MODEL_PATH = os.path.join(DRIVE_PATH, "models", "best_genre_model_gtzan.pth")

# Run cross evaluation
!python scripts/cross_evaluate.py \
    --model_path "{MODEL_PATH}" \
    --csv_file "{JAMENDO_CSV}" \
    --base_dir "{JAMENDO_BASE}" \
    --batch_size 64 \
    --num_workers 2